# S1-4. AlphaGo, MCTS와 지휘결심 지원 시스템 (온라인 보충)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2038prob/defense-ai-textbook-code/blob/main/notebooks/online/%EC%98%A8%EB%9D%BC%EC%9D%B8%EB%B3%B4%EC%B6%A9_S1-4_AlphaGo_MCTS%EC%99%80_%EC%A7%80%ED%9C%98%EA%B2%B0%EC%8B%AC_%EC%A7%80%EC%9B%90_%EC%8B%9C%EC%8A%A4%ED%85%9C.ipynb)

「국방 인공지능」 실습 코드 — 이 장의 파이썬 코드를 순서대로 모았습니다. **맨 위 setup 셀부터 위에서 아래로** 실행하세요(앞 셀에서 정의한 변수를 뒤 셀이 이어 씁니다). ⚠️ 표시가 붙은 셀은 개념 설명용 조각으로, 그대로는 실행되지 않을 수 있습니다.

전체 저장소: https://github.com/2038prob/defense-ai-textbook-code


In [ ]:
# === 실습 환경 준비 (Colab/로컬 공통) — 가장 먼저 실행하세요 ===
# 필요한 패키지를 설치하고(이미 있으면 건너뜀) 공통 유틸을 정의합니다.
import importlib.util, subprocess, sys
for pkg in ["torch", "torchvision", "numpy", "matplotlib", "scipy"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

## 보상·제약·차폐의 3계층 아키텍처


> ⚠️ **설명용 조각** — 아래 `shield` 함수는 차폐(safety layer) 개념을 보여주는 의사코드입니다. `rule.violated`, `log_intervention` 등은 이 노트북에 구현되어 있지 않으므로 호출하면 오류가 납니다(정의만 하는 이 셀은 그대로 실행됩니다).

In [ ]:
def shield(proposed_action, state, rules):
    """실행 직전 안전 검사: 위반이면 안전 대안으로 대체하고 기록한다."""
    for rule in rules:                      # 예: 고도 하한, 금지 구역, 교전 규칙
        if rule.violated(state, proposed_action):
            safe = rule.fallback(state)     # 예: 상승, 경로 재계획, 사격 보류
            log_intervention(state, proposed_action, rule.name, safe)
            return safe                     # 정책 제안을 기각하고 대안 실행
    return proposed_action                  # 모든 규칙 통과 시 원안 실행

## 실습: 격자월드에서 MCTS 구현하기

### 🔬 실습 — 순수 Python MCTS

① 환경과 전이 함수를 정의한다.


In [ ]:
import random, math

W, H = 5, 5
START, GOAL = (0, 0), (4, 4)
BLOCK = {(1, 1), (1, 2), (3, 2)}
ACTIONS = [(0,-1), (0,1), (-1,0), (1,0)]
NAMES = ["상", "하", "좌", "우"]

def step(s, a_idx):
    x, y = s
    dx, dy = ACTIONS[a_idx]
    ns = (max(0, min(W-1, x+dx)), max(0, min(H-1, y+dy)))
    if ns in BLOCK:
        ns, r = s, -2.0
    elif ns == GOAL:
        r = 10.0
    else:
        r = -0.1
    return ns, r, (ns == GOAL)

② 노드와 UCB 점수를 정의한다.


In [ ]:
class Node:
    def __init__(self, state, parent=None, action=None):
        self.state = state
        self.parent = parent
        self.action = action
        self.children = {}
        self.N = 0
        self.W = 0.0
    @property
    def Q(self):
        return self.W / max(1, self.N)
    def ucb(self, c=1.4):
        if self.N == 0:
            return float("inf")
        return self.Q + c * math.sqrt(math.log(self.parent.N + 1) / self.N)

③ rollout과 MCTS 루프를 구현한다.


In [ ]:
def rollout(s, depth=20):
    total = 0.0
    for _ in range(depth):
        a = random.randrange(len(ACTIONS))
        s, r, done = step(s, a)
        total += r
        if done:
            break
    return total

def mcts(root_state, sims=500):
    root = Node(root_state)
    for _ in range(sims):
        node = root
        # 1. selection
        while len(node.children) == len(ACTIONS) and node.children:
            node = max(node.children.values(), key=lambda n: n.ucb())
        # 2. expansion
        if node.state != GOAL:
            untried = [a for a in range(len(ACTIONS)) if a not in node.children]
            if untried:
                a = random.choice(untried)
                ns, r, done = step(node.state, a)
                child = Node(ns, parent=node, action=a)
                node.children[a] = child
                node = child
        # 3. simulation
        value = rollout(node.state)
        # 4. backup
        while node is not None:
            node.N += 1
            node.W += value
            node = node.parent
    return root

④ 루트에서 추천 행동을 확인한다.


In [ ]:
random.seed(7)
root = mcts(START, sims=500)
for a, child in sorted(root.children.items()):
    print(NAMES[a], "N=", child.N, "Q=", round(child.Q, 2), "state=", child.state)

best = max(root.children.values(), key=lambda n: n.N)
print("추천 행동:", NAMES[best.action])

**실행 결과 예시**

```
상 N= 3 Q= -8.97 state= (0, 0)
하 N= 2 Q= -9.6 state= (0, 1)
좌 N= 2 Q= -11.5 state= (0, 0)
우 N= 493 Q= 4.93 state= (1, 0)
추천 행동: 우
```


---

실습 코드 저장소: https://github.com/2038prob/defense-ai-textbook-code  ·  교재 「국방 인공지능」
